In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict
import pandas as pd
import nltk

def load_texts(texts_dir):
    texts = {}
    for txt_file in Path(texts_dir).glob("*.txt"):
        texts[txt_file.stem] = txt_file.read_text(encoding="utf-8")
    return texts

def load_entities(entity_tsv):
    df = pd.read_csv(entity_tsv, sep="\t")
    entities_by_doc = {}

    for _, row in df.iterrows():
        doc_id = str(row["document_id"])
        ent = {
            "type": str(row["entity_type"]),
            "text": str(row["entity_text"]),
            "span": str(row["entity_span"]),
        }
        entities_by_doc.setdefault(doc_id, []).append(ent)

    return entities_by_doc

def load_relations(rel_tsv):
    return pd.read_csv(rel_tsv, sep="\t")


def parse_span(span_str):
    start, end = str(span_str).split("-")
    return int(start), int(end)

def span_distance(span1, span2):
    s1, e1 = parse_span(span1)
    s2, e2 = parse_span(span2)

    if max(s1, s2) <= min(e1, e2):
        return 0
    if e1 < s2:
        return s2 - e1
    return s1 - e2

def get_sentence_spans(text, lang="english"):
    try:
        sent_tokenizer = nltk.data.load(f"tokenizers/punkt/{lang}.pickle")
    except LookupError:
        sent_tokenizer = nltk.data.load("tokenizers/punkt/english.pickle")
    return list(sent_tokenizer.span_tokenize(text))

def get_sentence_index(sentence_spans, span):
    start, end = parse_span(span)
    for i, (s_start, s_end) in enumerate(sentence_spans):
        if s_start <= start < s_end and s_start < end <= s_end:
            return i
    return None

def in_same_sentence(text, span1, span2, lang="english"):
    sentence_spans = get_sentence_spans(text, lang=lang)
    idx1 = get_sentence_index(sentence_spans, span1)
    idx2 = get_sentence_index(sentence_spans, span2)
    return idx1 is not None and idx2 is not None and idx1 == idx2

def find_sentence_segment(text, head_start, head_end, tail_start, tail_end, lang="english"):
    sentence_spans = get_sentence_spans(text, lang=lang)
    if not sentence_spans:
        return text, 0

    entity_min = min(head_start, tail_start)
    entity_max = max(head_end, tail_end)

    first_idx = 0
    last_idx = len(sentence_spans) - 1

    for i, (s_start, s_end) in enumerate(sentence_spans):
        if s_start <= entity_min < s_end:
            first_idx = i
        if s_start < entity_max <= s_end:
            last_idx = i

    seg_start = sentence_spans[first_idx][0]
    seg_end = sentence_spans[last_idx][1]
    return text[seg_start:seg_end], seg_start

def make_instance(text, head_ent, tail_ent, relation, doc_id, lang="english"):
    head_start, head_end = parse_span(head_ent["span"])
    tail_start, tail_end = parse_span(tail_ent["span"])

    segment, offset = find_sentence_segment(
        text, head_start, head_end, tail_start, tail_end, lang=lang
    )

    h_start = head_start - offset
    h_end = head_end - offset
    t_start = tail_start - offset
    t_end = tail_end - offset

    if h_start < 0 or h_end > len(segment) or t_start < 0 or t_end > len(segment):
        segment = text
        h_start, h_end = head_start, head_end
        t_start, t_end = tail_start, tail_end

    return {
        "text": segment,
        "h": {"name": head_ent["text"], "pos": [h_start, h_end]},
        "t": {"name": tail_ent["text"], "pos": [t_start, t_end]},
        "relation": relation,
        "doc_id": doc_id,
        "head_span": head_ent["span"],
        "tail_span": tail_ent["span"],
        "head_type": head_ent["type"],
        "tail_type": tail_ent["type"],
    }


def build_valid_type_pairs_from_relations(rel_df, no_relation_label="no_relation"):
    valid_pairs = set()
    for _, row in rel_df.iterrows():
        rel = str(row["relation"])
        if rel == no_relation_label:
            continue
        valid_pairs.add((str(row["head_type"]), str(row["tail_type"])))
    return valid_pairs


def generate_same_sentence_hard_negatives(
    rel_tsv,
    entity_tsv,
    texts_dir,
    output_path,
    lang="english",
    neg_ratio=1,
    no_relation_label="no_relation",
    only_valid_pair_types=True,
    sample_closest_first=True,
    seed=42,
):
    rng = random.Random(seed)

    rel_df = load_relations(rel_tsv)
    entities_by_doc = load_entities(entity_tsv)
    texts = load_texts(texts_dir)

    valid_type_pairs = build_valid_type_pairs_from_relations(rel_df, no_relation_label=no_relation_label)

    positive_pairs_by_doc = defaultdict(set)
    positive_instances = []

    for _, row in rel_df.iterrows():
        doc_id = str(row["document_id"])

        head_ent = {
            "type": str(row["head_type"]),
            "text": str(row["head_text"]),
            "span": str(row["head_span"]),
        }
        tail_ent = {
            "type": str(row["tail_type"]),
            "text": str(row["tail_text"]),
            "span": str(row["tail_span"]),
        }

        positive_pairs_by_doc[doc_id].add((head_ent["span"], tail_ent["span"]))

        if doc_id in texts:
            positive_instances.append(
                make_instance(
                    text=texts[doc_id],
                    head_ent=head_ent,
                    tail_ent=tail_ent,
                    relation=str(row["relation"]),
                    doc_id=doc_id,
                    lang=lang,
                )
            )

    negative_instances = []

    for doc_id, entities in entities_by_doc.items():
        if doc_id not in texts:
            continue

        text = texts[doc_id]
        pos_set = positive_pairs_by_doc.get(doc_id, set())

        candidates = []

        for i, head_ent in enumerate(entities):
            for j, tail_ent in enumerate(entities):
                if i == j:
                    continue

                pair_key = (head_ent["span"], tail_ent["span"])

                if pair_key in pos_set:
                    continue

                if only_valid_pair_types and (head_ent["type"], tail_ent["type"]) not in valid_type_pairs:
                    continue

                if not in_same_sentence(text, head_ent["span"], tail_ent["span"], lang=lang):
                    continue

                dist = span_distance(head_ent["span"], tail_ent["span"])

                candidates.append((dist, head_ent, tail_ent))

        if sample_closest_first:
            candidates = sorted(candidates, key=lambda x: x[0])
        else:
            rng.shuffle(candidates)

        n_pos_doc = len(pos_set)
        n_take = min(len(candidates), neg_ratio * n_pos_doc)

        chosen = candidates[:n_take]

        for dist, head_ent, tail_ent in chosen:
            negative_instances.append(
                make_instance(
                    text=text,
                    head_ent=head_ent,
                    tail_ent=tail_ent,
                    relation=no_relation_label,
                    doc_id=doc_id,
                    lang=lang,
                )
            )

    all_instances = positive_instances + negative_instances

    with open(output_path, "w", encoding="utf-8") as f:
        for inst in all_instances:
            f.write(json.dumps(inst, ensure_ascii=False) + "\n")

    print(f"Saved {len(all_instances)} instances to {output_path}")
    print(f"  positives: {len(positive_instances)}")
    print(f"  negatives: {len(negative_instances)}")


In [ ]:
generate_same_sentence_hard_negatives(
    rel_tsv="data/en/train/eng-train-rel.tsv",
    entity_tsv="data/en/train/eng-train-ent.tsv",
    texts_dir="data/en/train/texts",
    output_path="train_with_hard_negatives.jsonl",
    lang="english",
    neg_ratio=2,
    no_relation_label="no_relation",
    only_valid_pair_types=True,
    sample_closest_first=True,
    seed=42,
)


Saved 9557 instances to train_with_hard_negatives.jsonl
  positives: 3193
  negatives: 6364
